# Tema 7 · Discretización de atributos numéricos

**Trabajo de investigación aplicada · Minería de Datos · grupal**

**Integrantes del grupo:** _escriba aquí los nombres_

## 📖 El caso

**Termas Los Coigües** es un centro termal de la precordillera, en Pinto. Tienen el registro de 800 visitantes del verano pasado: su edad, cuánto gastaron, cuántas noches se quedaron, de qué distancia venían y si **volvieron** este año. El área comercial quiere armar una **campaña de fidelización por tramos de gasto**, pero no sabe dónde cortar los tramos: alguien propuso «cuatro tramos iguales» y otro, «por cuartiles».

> **❓ La pregunta del cliente:** ¿Cómo conviene dividir el gasto en tramos para entender quién vuelve y a quién dirigir la campaña?

## 🗂️ Los datos: `t7_termas.csv` (800 visitantes con edad, gasto, noches, distancia y si volvió)

| Columna | Qué significa |
|---|---|
| `visitante` | número del visitante |
| `edad` | edad, en años |
| `gasto_miles` | gasto total de la visita, en miles de pesos |
| `noches` | noches de alojamiento (0 = solo el día) |
| `distancia_km` | distancia desde donde viene, en km |
| `vuelve` | **Sí / No**: si volvió esta temporada ← lo que se quiere predecir |

## Cómo trabajar este notebook

1. **Archivo → Guardar una copia en Drive.** Trabajen siempre en esa copia. Para trabajar en grupo, el dueño de la copia la **comparte** con el resto (botón **Compartir**).
2. **Todo el código viene escrito y explicado.** No hay que programar desde cero: lean la explicación 📘 que está antes de cada celda y ejecútenla (▶) **en orden**.
3. Las líneas **`# ✅ Debe mostrar:`** dicen qué debe aparecer. Si aparece otra cosa, hagan **Entorno de ejecución → Reiniciar y ejecutar todo**.
4. En las celdas **✏️ Experimento** cambien el valor que se indica, vuelvan a ejecutar y anoten lo que pasa en la tabla de abajo. **Al terminar, dejen el valor original.**
5. Respondan cada **✍️** con **sus números**. Esas respuestas son la materia prima de la **parte práctica del informe** y de la **exposición**.

| Ícono | Qué es |
|:-:|---|
| 📘 | Explicación de lo que hace la celda siguiente |
| ✅ | Lo que debe mostrar la celda |
| ✏️ | Experimento: ustedes cambian un valor |
| ✍️ | Pregunta para responder con sus números |
| 🧠 | De la información al conocimiento: convertir el resultado en una decisión para el cliente |

In [ ]:
# ▶ CELDA DE INICIO — no la modifique. Importa las librerías y carga los datos.
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

RUTA = "https://raw.githubusercontent.com/de-mag-ia/diguillin-datos/main/investigacion/"
datos = pd.read_csv(RUTA + "t7_termas.csv")

print(f"Datos cargados: {len(datos)} filas y {datos.shape[1]} columnas.")
# ✅ Debe mostrar: Datos cargados: 800 filas y 6 columnas.

## 1. Conocer los datos

📘 Cada fila es un **visitante** de las termas el verano pasado: su edad, cuánto gastó (en miles de pesos), cuántas
noches se quedó, de qué distancia venía y si **volvió** este año. La celda cuenta cuántos volvieron y muestra el
rango del gasto.

In [ ]:
datos.head()

In [ ]:
print("Volvieron:", (datos["vuelve"] == "Sí").sum(), "de", len(datos))
print("Gasto (miles de $): mínimo", datos["gasto_miles"].min(),
      "· mediana", datos["gasto_miles"].median(), "· máximo", datos["gasto_miles"].max())
# ✅ Debe mostrar: Volvieron: 478 de 800  /  Gasto (miles de $): mínimo 15 · mediana 120.0 · máximo 9500

### ✍️ Comparen la mediana del gasto con el máximo. ¿Qué les llama la atención? Busquen ese visitante en la tabla (`datos.sort_values("gasto_miles").tail()`): ¿es un error o un caso real?

_Su respuesta:_

## 2. Separar entrenamiento y prueba, y la línea base

📘 Se separa el 75 % para **construir** las reglas y el 25 % para **probarlas**. La línea base es **ZeroR**: decir
siempre la respuesta más común.

In [ ]:
ent, pru = train_test_split(datos, test_size=0.25, random_state=2026, stratify=datos["vuelve"])
clase_comun = ent["vuelve"].value_counts().idxmax()
print("Entrenamiento:", len(ent), "· Prueba:", len(pru))
print(f"ZeroR (decir siempre '{clase_comun}'): {(pru['vuelve'] == clase_comun).mean():.0%}")
# ✅ Debe mostrar: Entrenamiento: 600 · Prueba: 200  /  ZeroR (decir siempre 'Sí'): 60%

## 3. OneR necesita tramos

📘 **OneR** (Semana 6) arma una regla del tipo «si el atributo vale X, entonces…». Con un atributo numérico como el
gasto eso no funciona: casi cada visitante tiene un gasto distinto. Primero hay que **discretizar**: convertir el
número en **tramos** («menos de 50 mil», «de 50 a 150 mil»…). Después, en cada tramo, OneR predice la respuesta más
común en entrenamiento.

La función `oner_por_tramos()` hace exactamente eso y mide el acierto en prueba. Recibe el número de tramo de cada
visitante.

In [ ]:
def oner_por_tramos(tramo, nombre):
    """OneR: en cada tramo predice la respuesta más común en ENTRENAMIENTO y mide el acierto en PRUEBA."""
    regla = pd.crosstab(tramo[ent.index], ent["vuelve"]).idxmax(axis=1)   # tramo → 'Sí' o 'No'
    prediccion = tramo[pru.index].map(regla).fillna(clase_comun)           # tramo sin datos → la clase común
    acierto = (prediccion == pru["vuelve"]).mean()
    print(f"{nombre}: acierto en prueba {acierto:.0%}")
    return regla

## 4. Tramos de igual ancho

📘 `pd.cut(..., bins=4)` divide el rango **del mínimo al máximo** en 4 tramos **del mismo ancho**. Es lo más
simple. La celda muestra los puntos de corte y cuántos visitantes caen en cada tramo.

In [ ]:
tramo_ancho, limites = pd.cut(datos["gasto_miles"], bins=4, labels=False, retbins=True)
print("Cortes (miles de $):", " · ".join(f"{x:.0f}" for x in limites[1:-1]))
print("Visitantes por tramo:", np.bincount(tramo_ancho, minlength=4).tolist())
oner_por_tramos(tramo_ancho, "OneR con igual ancho")
# ✅ Debe mostrar: Cortes (miles de $): 2386 · 4758 · 7129  /  Visitantes por tramo: [799, 0, 0, 1]  /  OneR con igual ancho: acierto en prueba 60%

📘 Una forma de proteger los tramos de igual ancho es **recortar** el atípico: los gastos sobre el percentil 99
se reemplazan por el valor del percentil 99 (`clip`). El visitante no se borra; solo deja de estirar la escala.

In [ ]:
tope = datos["gasto_miles"].quantile(0.99)
tramo_recortado, limites_r = pd.cut(datos["gasto_miles"].clip(upper=tope), bins=4, labels=False, retbins=True)
print(f"Tope (percentil 99): {tope:.0f}")
print("Visitantes por tramo:", np.bincount(tramo_recortado, minlength=4).tolist())
oner_por_tramos(tramo_recortado, "OneR con igual ancho, recortado")
# ✅ Debe mostrar: Tope (percentil 99): 634  /  Visitantes por tramo: [543, 194, 41, 22]  /  OneR con igual ancho, recortado: acierto en prueba 68%

### ✍️ ¿Qué le pasó a los tramos de igual ancho con los datos originales y por qué OneR quedó igual que ZeroR? ¿Qué cambió al recortar el atípico?

_Su respuesta:_

## 5. Tramos de igual frecuencia

📘 `pd.qcut(..., q=4)` hace tramos con **la misma cantidad de visitantes** (cuartiles; no queda exacta porque
varios visitantes tienen el mismo gasto y no se pueden separar). No le importan los atípicos, porque corta por
posición y no por valor.

In [ ]:
tramo_frec, limites_q = pd.qcut(datos["gasto_miles"], q=4, labels=False, retbins=True)
print("Cortes (miles de $):", " · ".join(f"{x:.0f}" for x in limites_q[1:-1]))
print("Visitantes por tramo:", np.bincount(tramo_frec, minlength=4).tolist())
oner_por_tramos(tramo_frec, "OneR con igual frecuencia")
# ✅ Debe mostrar: Cortes (miles de $): 71 · 120 · 198  /  Visitantes por tramo: [203, 198, 199, 200]  /  OneR con igual frecuencia: acierto en prueba 65%

## 6. Discretización supervisada: cortar donde cambia la respuesta

📘 Los dos métodos anteriores cortan **sin mirar** si el visitante volvió o no (son **no supervisados**). La
discretización **supervisada** usa la respuesta: busca los cortes que mejor separan a los que vuelven de los que no.
Una forma simple es pedirle a un **árbol de decisión** (Semana 7) que use solo el gasto y que tenga como máximo 4
hojas: sus preguntas («¿gasto ≤ 49,5?») son los puntos de corte. Se entrena **solo** con los datos de entrenamiento.

In [ ]:
arbol = DecisionTreeClassifier(max_leaf_nodes=4, random_state=0).fit(ent[["gasto_miles"]], ent["vuelve"])
umbrales = sorted(u for u in arbol.tree_.threshold if u != -2)     # -2 marca las hojas, que no cortan
print("Cortes encontrados por el árbol:", " · ".join(f"{u:.1f}" for u in umbrales))

tramo_sup = pd.Series(np.digitize(datos["gasto_miles"], umbrales), index=datos.index)
for k in range(4):
    del_tramo = datos[tramo_sup == k]
    print(f"Tramo {k + 1}: {len(del_tramo)} visitantes, vuelve el {(del_tramo['vuelve'] == 'Sí').mean():.0%}")
oner_por_tramos(tramo_sup, "OneR con tramos supervisados")
# ✅ Debe mostrar: Cortes encontrados por el árbol: 49.5 · 148.5 · 277.5  /  Tramo 1: 102 visitantes, vuelve el 19%  /  Tramo 2: 386 visitantes, vuelve el 82%  /  Tramo 3: 215 visitantes, vuelve el 59%  /  Tramo 4: 97 visitantes, vuelve el 16%  /  OneR con tramos supervisados: acierto en prueba 80%

### ✍️ Comparen los tres OneR (igual ancho, igual frecuencia y supervisado) con ZeroR. ¿Cuál gana y por qué? Describan en palabras qué tipo de visitante vuelve más.

_Su respuesta:_

## 7. ¿Cuántos tramos?

📘 Con igual ancho e igual frecuencia, el analista elige **cuántos** tramos usar. Prueben varios.

In [ ]:
# ✏️ EXPERIMENTO — cambie el valor indicado y vuelva a ejecutar
TRAMOS = 4        # ✏️ pruebe con 2, 3, 4, 5, 6, 7, 8, 9 y 10 (al terminar, deje 4)

oner_por_tramos(pd.cut(datos["gasto_miles"], bins=TRAMOS, labels=False), f"Igual ancho, {TRAMOS} tramos")
oner_por_tramos(pd.qcut(datos["gasto_miles"], q=TRAMOS, labels=False), f"Igual frecuencia, {TRAMOS} tramos")

**✏️ Anoten aquí sus resultados:**

| Tramos | OneR con igual ancho | OneR con igual frecuencia |
|:-:|:-:|:-:|
| 2 | | |
| 3 | | |
| 4 | | |
| 5 | | |
| 6 | | |
| 7 | | |
| 8 | | |
| 9 | | |
| 10 | | |

### ✍️ Según su tabla, ¿mejora alguna vez el igual ancho? ¿Con cuántos tramos el igual frecuencia se acerca al supervisado? ¿Qué costo tiene usar muchos tramos?

_Su respuesta:_

## 8. Tramos con sentido de negocio: la edad

📘 A veces los tramos no los elige un algoritmo sino el **negocio**: para una campaña se usan tramos que el área de
marketing entiende. La celda corta la edad en cuatro tramos clásicos y calcula cuántos vuelven en cada uno.

In [ ]:
tramo_edad = pd.cut(datos["edad"], bins=[17, 29, 44, 59, 80],
                    labels=["18 a 29", "30 a 44", "45 a 59", "60 o más"])
for tramo, grupo in datos.groupby(tramo_edad, observed=True):
    print(f"{tramo}: vuelve el {(grupo['vuelve'] == 'Sí').mean():.0%}")
# ✅ Debe mostrar: 18 a 29: vuelve el 54%  /  30 a 44: vuelve el 62%  /  45 a 59: vuelve el 55%  /  60 o más: vuelve el 65%

### ✍️ 🧠 **De la información al conocimiento.** Las termas quieren lanzar una **campaña de fidelización** (descuento para volver). Escriban la recomendación: ¿a qué tramos de gasto (y de edad) apuntar, cuáles cortes usar y por qué no conviene usar tramos de igual ancho?

_Su respuesta:_

---
## ✔️ Para entregar

1. Revisen que **todas** las ✍️ tengan respuesta y que las tablas ✏️ estén completas.
2. **Entorno de ejecución → Reiniciar y ejecutar todo**: todo debe correr sin errores.
3. **Archivo → Descargar → Descargar .ipynb** y súbanlo al **Aula Virtual** como `Grupo_N_Tema7.ipynb`, junto con el informe en PDF.